# 08 — Prepare OncoKB annotation inputs

This notebook converts the ANNOVAR-annotated variant table into tab-delimited inputs for the OncoKB MafAnnotator.

It:

- preserves all original variant and annotation columns;
- uses the original anchor-preserved variant coordinates and alleles;
- creates the required MAF-style columns;
- constructs `HGVSg`;
- formats `HGVSp` for OncoKB;
- saves separate inputs for all variants, ClinVar-pathogenic target-gene variants, and ClinVar-benign target-gene variants.

This notebook only prepares inputs. Completed OncoKB outputs are examined and merged in a later notebook.

In [ ]:
from pathlib import Path
import pandas as pd

from pathlib import Path
import pandas as pd

notebook_dir = Path("/home/donetski/Notebooks")
input_csv = notebook_dir / "OutputFiles" / "06_annovar_output" / "fixed_annovar_output.csv"
output_dir = notebook_dir / "OutputFiles" / "08_oncokb_input"
output_dir.mkdir(parents=True, exist_ok=True)

output_prefix = "08"
all_variants_output = output_dir / f"{output_prefix}_oncokb_input_all_variants.txt"
pathogenic_output = output_dir / f"{output_prefix}_oncokb_input_pathogenic_target_genes.txt"
benign_output = output_dir / f"{output_prefix}_oncokb_input_benign_target_genes.txt"

TARGET_GENES = ["ATM", "BARD1", "BRCA1", "BRCA2", "CDH1", "CDKN2A", "CHEK2", "MLH1", "MSH2", "MSH6", "PALB2", "PMS2", "PTEN", "RAD51C", "RAD51D", "TP53"]
CLINVAR_PATHOGENIC = {"Pathogenic", "Likely_pathogenic", "Pathogenic/Likely_pathogenic"}
CLINVAR_BENIGN = {"Benign", "Likely_benign", "Benign/Likely_benign"}

## Load the annotated variant table

The file is loaded as strings so allele, coordinate, HGVS, and annotation values retain their original formatting.

In [ ]:
df = pd.read_csv(input_csv, dtype=str, low_memory=False).fillna(".")
print(f"Rows: {len(df):,} | Samples: {df['Sample.ID'].nunique():,}")

## Confirm the source variant fields

OncoKB coordinates are built from the original anchor-preserved variant fields rather than the normalized ANNOVAR `Chr`, `Start`, `Ref`, and `Alt` columns.

In [ ]:
source_cols = ["Sample.ID", "Gene", "HGVSp", "Chr.1", "Start.1", "REF", "ALT", "ClinVar.SIG"]
df[source_cols].head()

## Construct the OncoKB input columns

The original anchor-preserved `REF` and `ALT` fields are used to reconstruct each genomic variant. `End_Position` includes the full anchored reference allele.

In [ ]:
def prepare_oncokb_input(variants):
    out = variants.copy()
    chrom = out["Chr.1"].str.replace("chr", "", case=False, regex=False)
    start, ref, alt = out["Start.1"].astype(int), out["REF"], out["ALT"]
    end = start + ref.str.len() - 1

    out["NCBI_Build"] = "GRCh38"
    out["Hugo_Symbol"] = out["Gene"]
    out["Tumor_Sample_Barcode"] = out["Sample.ID"]
    out["Chromosome"], out["Start_Position"], out["End_Position"] = chrom, start, end
    out["Reference_Allele"], out["Tumor_Seq_Allele2"] = ref, alt
    out["HGVSg"] = [f"{c}:g.{s}{r}>{a}" if s == e else f"{c}:g.{s}_{e}{r}>{a}" for c, s, e, r, a in zip(chrom, start, end, ref, alt)]
    out["HGVSp_original"] = out["HGVSp"]
    out["HGVSp"] = out["HGVSp"].str.replace(r"^p\.", "", regex=True).str.replace("Ter", "*", regex=False)

    front_cols = ["NCBI_Build", "Hugo_Symbol", "Tumor_Sample_Barcode", "HGVSp", "HGVSg", "Chromosome", "Start_Position", "End_Position", "Reference_Allele", "Tumor_Seq_Allele2"]
    return out[front_cols + [col for col in out.columns if col not in front_cols]]

In [ ]:
all_oncokb = prepare_oncokb_input(df)

pathogenic_df = df[df["ClinVar.SIG"].isin(CLINVAR_PATHOGENIC) & df["Gene"].isin(TARGET_GENES)].copy()
pathogenic_oncokb = prepare_oncokb_input(pathogenic_df)

benign_df = df[df["ClinVar.SIG"].isin(CLINVAR_BENIGN) & df["Gene"].isin(TARGET_GENES)].copy()
benign_oncokb = prepare_oncokb_input(benign_df)

print(f"All variants: {len(all_oncokb):,}")
print(f"Pathogenic target-gene variants: {len(pathogenic_oncokb):,}")
print(f"Benign target-gene variants: {len(benign_oncokb):,}")

In [ ]:
all_oncokb.to_csv(all_variants_output, sep="\t", index=False)
pathogenic_oncokb.to_csv(pathogenic_output, sep="\t", index=False)
benign_oncokb.to_csv(benign_output, sep="\t", index=False)

print(all_variants_output)
print(pathogenic_output)
print(benign_output)